# Eye 02

Eye-tracking / pupil analysis (part 2).

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

columns_mapping = {
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
}

baseline_eye_tracking = pd.read_csv(f'{DATA}/sed.csv').rename(columns=columns_mapping)
eye_tracking_01 = pd.read_csv(f'{DATA}/sed_01.csv').rename(columns=columns_mapping)
eye_tracking_02 = pd.read_csv(f'{DATA}/sed_02.csv').rename(columns=columns_mapping)
eye_tracking_03 = pd.read_csv(f'{DATA}/sed_03.csv').rename(columns=columns_mapping)

psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

def clean_eye_tracking_data(eye_tracking_data):
    df = eye_tracking_data.copy()
    df['timestamp'] = pd.to_datetime(
        df['timestamp'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    return df.dropna(subset=['timestamp'])

baseline_eye_tracking = clean_eye_tracking_data(baseline_eye_tracking)
eye_tracking_01 = clean_eye_tracking_data(eye_tracking_01)
eye_tracking_02 = clean_eye_tracking_data(eye_tracking_02)
eye_tracking_03 = clean_eye_tracking_data(eye_tracking_03)

psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

def filter_eye_tracking_data(eye_tracking_data, questions):
    start_time = questions['Question Start Time'].min()
    end_time = questions['Question Answer Time'].max()
    return eye_tracking_data[
        (eye_tracking_data['timestamp'] >= start_time) &
        (eye_tracking_data['timestamp'] <= end_time)
    ]

BLINK_THRESHOLD = 1.0
MIN_CLOSED_FRAMES = 3  # ~100ms at 30Hz

def count_blinks(series, threshold, min_frames):
    # sustained closure onset
    closed = (series <= threshold).astype(int)
    sustained = closed.rolling(min_frames).sum() == min_frames
    return int((sustained & ~sustained.shift(1, fill_value=False)).sum())

def calculate_eye_tracking_metrics(eye_tracking_data):
    average_pupil_dilation = eye_tracking_data['pupil_dilation'].mean()
    total_duration_minutes = (
        eye_tracking_data['timestamp'].max() - eye_tracking_data['timestamp'].min()
    ).total_seconds() / 60
    if total_duration_minutes <= 0:
        return average_pupil_dilation, 0.0, 0.0
    left_blink_rate = count_blinks(eye_tracking_data['left_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes
    right_blink_rate = count_blinks(eye_tracking_data['right_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes
    return average_pupil_dilation, left_blink_rate, right_blink_rate

def detect_significant_increase(test_metrics, baseline_metrics):
    return (
        test_metrics[0] > baseline_metrics[0],
        test_metrics[1] > baseline_metrics[1],
        test_metrics[2] > baseline_metrics[2]
    )

baseline_metrics = calculate_eye_tracking_metrics(baseline_eye_tracking)

question_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']

print("Setup complete. All data loaded and cleaned.")


In [ ]:
psychometric_data = {
    qt: [
        psychometric_01[psychometric_01['Type'] == qt],
        psychometric_02[psychometric_02['Type'] == qt],
        psychometric_03[psychometric_03['Type'] == qt]
    ]
    for qt in question_types
}

results = []
for question_type in question_types:
    for i, (questions, eye_data) in enumerate(zip(
            psychometric_data[question_type],
            [eye_tracking_01, eye_tracking_02, eye_tracking_03]), 1):
        filtered = filter_eye_tracking_data(eye_data, questions)
        metrics = calculate_eye_tracking_metrics(filtered)
        sig = detect_significant_increase(metrics, baseline_metrics)
        results.append({
            'Test': f'Test {i:02d}',
            'Type': question_type,
            'Start Time': questions['Question Start Time'].min().strftime('%H:%M:%S'),
            'End Time': questions['Question Answer Time'].max().strftime('%H:%M:%S'),
            'Average Pupil Dilation': round(metrics[0], 2),
            'Left Blink Rate': round(metrics[1], 2),
            'Right Blink Rate': round(metrics[2], 2),
            'Significant Increase (Pupil Dilation)': 'Yes' if sig[0] else 'No',
            'Significant Increase (Left Blink Rate)': 'Yes' if sig[1] else 'No',
            'Significant Increase (Right Blink Rate)': 'Yes' if sig[2] else 'No',
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

for metric, ylabel in [
    ('Average Pupil Dilation', 'Pupil Dilation'),
    ('Left Blink Rate', 'Left Blink Rate (blinks/min)'),
    ('Right Blink Rate', 'Right Blink Rate (blinks/min)')
]:
    plt.figure(figsize=(14, 6))
    for qt in question_types:
        subset = results_df[results_df['Type'] == qt]
        plt.plot(subset['Test'], subset[metric], marker='o', label=qt)
    plt.title(f'{metric} by Test and Question Type')
    plt.xlabel('Test')
    plt.ylabel(ylabel)
    plt.legend(title='Question Type')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()